# Inspect Development Pipeline Results (Functional)

This notebook loads generated CSV outputs, validates file availability, and provides reusable analysis functions for performance, shuffled-label controls, stability, and SHAP feature inspection.

This version is written to be modular and easy to extend for future result-checking or final thesis evaluation workflows.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUTS = PROJECT_ROOT / "outputs"
FIGURES = OUTPUTS / "figures"

files = {
    "features": OUTPUTS / "features.csv",
    "baseline": OUTPUTS / "baseline_results.csv",
    "optimized": OUTPUTS / "optimized_results.csv",
    "shuffled": OUTPUTS / "shuffled_label_results.csv",
    "shap_baseline": OUTPUTS / "shap_values_baseline.csv",
    "shap_optimized": OUTPUTS / "shap_values_optimized.csv",
    "stability": OUTPUTS / "stability_results.csv",
}

def check_files(file_map):
    status = []
    for name, path in file_map.items():
        status.append({
            "name": name,
            "path": str(path),
            "exists": path.exists(),
        })
    return pd.DataFrame(status)

def load_csv_if_exists(path):
    if not path.exists():
        raise FileNotFoundError(f'Expected file not found: {path}')
    return pd.read_csv(path)

print("Project root:", PROJECT_ROOT)
print("Outputs directory:", OUTPUTS)
print()
display(check_files(files))

print("Figures:")
for figure_path in sorted(FIGURES.glob('*')):
    print(figure_path.name)

Project root: c:\Users\Yazan\Thesis
Outputs directory: c:\Users\Yazan\Thesis\outputs



,name,path,exists
0,features,c:\Users\Yazan\Thesis\outputs\features.csv,True
1,baseline,c:\Users\Yazan\Thesis\outputs\baseline_results...,True
2,optimized,c:\Users\Yazan\Thesis\outputs\optimized_result...,True
3,shuffled,c:\Users\Yazan\Thesis\outputs\shuffled_label_r...,True
4,shap_baseline,c:\Users\Yazan\Thesis\outputs\shap_values_base...,True
5,shap_optimized,c:\Users\Yazan\Thesis\outputs\shap_values_opti...,True
6,stability,c:\Users\Yazan\Thesis\outputs\stability_result...,True


Figures:
performance_comparison.png
stability_comparison.png
tradeoff_auroc_stability.png


## Load Data Tables

In [2]:
features = load_csv_if_exists(files['features'])
baseline = load_csv_if_exists(files['baseline'])
optimized = load_csv_if_exists(files['optimized'])
shuffled = load_csv_if_exists(files['shuffled'])
stability = load_csv_if_exists(files['stability'])

print('features shape:', features.shape)
display(features.head())

print('baseline shape:', baseline.shape)
display(baseline)

print('optimized shape:', optimized.shape)
display(optimized)

print('shuffled-label shape:', shuffled.shape)
display(shuffled)

print('stability shape:', stability.shape)
display(stability)

features shape: (98, 15)


,participant_id,cleaned_transcript,word_count,type_token_ratio,filler_count,filler_ratio,mean_clause_length,semantic_coherence,noun_ratio,verb_ratio,adjective_ratio,adverb_ratio,pronoun_ratio,determiner_ratio,diagnosis_label
0,Process-rec-006,fa a family in the kitchen mother washing up b...,118,0.6695,2.0000,0.0169,8.2167,0.2268,0.2167,0.1333,0.0083,0.0333,0.1250,0.1333,1
1,Process-rec-007,washing dishes and drying them the sink is ove...,204,0.5882,1.0000,0.0049,7.9465,0.1726,0.1898,0.1389,0.0370,0.0509,0.1389,0.1111,0
2,Process-rec-009,the picture is a kitchen in a house there's a ...,170,0.4824,5.0000,0.0294,11.0667,0.6124,0.3187,0.1099,0.0110,0.0055,0.0495,0.2143,0
3,Process-rec-010,the picture shows a mother and a boy and a gir...,292,0.4897,6.0000,0.0205,10.5750,0.1921,0.2273,0.1299,0.0390,0.0357,0.0877,0.1591,0
4,Process-rec-012,there is a woman by a si an overflowing sink w...,156,0.5192,8.0000,0.0513,8.5263,0.0000,0.2346,0.1173,0.0185,0.0000,0.1358,0.1605,0


baseline shape: (6, 30)


,model,variant,roc_auc_mean,roc_auc_std,accuracy_mean,accuracy_std,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,balanced_accuracy_mean,balanced_accuracy_std,average_precision_mean,average_precision_std,f2_mean,f2_std,recall_positive_mean,recall_positive_std,recall_negative_mean,recall_negative_std,tn_mean,tn_std,fp_mean,fp_std,fn_mean,fn_std,tp_mean,tp_std
0,logistic_regression,baseline,0.4298,0.2908,0.7863,0.0468,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4695,0.0198,0.2590,0.1511,0.0000,0.0000,0.0000,0.0000,0.9390,0.0396,15.4000,0.8000,1.0000,0.6325,3.2000,0.4000,0.0000,0.0000
1,logistic_regression_balanced,baseline,0.4721,0.2817,0.6121,0.0535,0.1844,0.1840,0.1333,0.1247,0.3167,0.3667,0.4933,0.1764,0.3037,0.2189,0.2439,0.2614,0.3167,0.3667,0.6699,0.0372,11.0000,0.8944,5.4000,0.4899,2.2000,1.1662,1.0000,1.0954
2,random_forest,baseline,0.5196,0.1390,0.8063,0.0366,0.0571,0.1143,0.0667,0.1333,0.0500,0.1000,0.5004,0.0339,0.2766,0.1157,0.0526,0.1053,0.0500,0.1000,0.9507,0.0466,15.6000,1.0198,0.8000,0.7483,3.0000,0.0000,0.2000,0.4000
3,random_forest_balanced,baseline,0.5169,0.1726,0.8163,0.0247,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4879,0.0149,0.2672,0.1393,0.0000,0.0000,0.0000,0.0000,0.9757,0.0297,16.0000,0.6325,0.4000,0.4899,3.2000,0.4000,0.0000,0.0000
4,svm_rbf,baseline,0.5162,0.1844,0.8368,0.0188,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000,0.0000,0.2495,0.0836,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,16.4000,0.4899,0.0000,0.0000,3.2000,0.4000,0.0000,0.0000
5,svm_rbf_balanced,baseline,0.5771,0.1586,0.7032,0.1031,0.1111,0.1405,0.1067,0.1373,0.1167,0.1453,0.4657,0.0943,0.2999,0.0887,0.1143,0.1432,0.1167,0.1453,0.8147,0.1135,13.4000,2.1541,3.0000,1.7889,2.8000,0.4000,0.4000,0.4899


optimized shape: (6, 31)


,model,variant,roc_auc_mean,roc_auc_std,accuracy_mean,accuracy_std,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,balanced_accuracy_mean,balanced_accuracy_std,average_precision_mean,average_precision_std,f2_mean,f2_std,recall_positive_mean,recall_positive_std,recall_negative_mean,recall_negative_std,tn_mean,tn_std,fp_mean,fp_std,fn_mean,fn_std,tp_mean,tp_std,best_params
0,logistic_regression,optimized,0.4958,0.2810,0.7968,0.0412,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4761,0.0221,0.2655,0.1558,0.0000,0.0000,0.0000,0.0000,0.9522,0.0442,15.6000,0.4899,0.8000,0.7483,3.2000,0.4000,0.0000,0.0000,"{'clf__C': 1.0, 'rfe__n_features_to_select': 4}"
1,logistic_regression_balanced,optimized,0.5112,0.2793,0.6121,0.0535,0.2067,0.1937,0.1452,0.1331,0.3667,0.3712,0.5120,0.1732,0.2824,0.1684,0.2788,0.2699,0.3667,0.3712,0.6574,0.0544,10.8000,1.1662,5.6000,0.8000,2.0000,1.0954,1.2000,1.1662,"{'clf__C': 0.5, 'rfe__n_features_to_select': 4}"
2,svm_rbf,optimized,0.4167,0.1793,0.8368,0.0188,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000,0.0000,0.1891,0.0471,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,16.4000,0.4899,0.0000,0.0000,3.2000,0.4000,0.0000,0.0000,"{'clf__C': 0.5, 'clf__gamma': 1.0}"
3,svm_rbf_balanced,optimized,0.6468,0.1638,0.7121,0.1115,0.0500,0.1000,0.0500,0.1000,0.0500,0.1000,0.4441,0.0777,0.3744,0.1254,0.0500,0.1000,0.0500,0.1000,0.8382,0.1293,13.8000,2.4819,2.6000,2.0591,3.0000,0.0000,0.2000,0.4000,"{'clf__C': 2.0, 'clf__gamma': 'scale'}"
4,random_forest,optimized,0.6157,0.1302,0.8063,0.0366,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4816,0.0150,0.3192,0.1152,0.0000,0.0000,0.0000,0.0000,0.9632,0.0300,15.8000,0.7483,0.6000,0.4899,3.2000,0.4000,0.0000,0.0000,"{'clf__max_depth': 4, 'clf__min_samples_leaf':..."
5,random_forest_balanced,optimized,0.6288,0.1380,0.7753,0.0702,0.0800,0.1600,0.1000,0.2000,0.0667,0.1333,0.4903,0.0799,0.3208,0.1642,0.0714,0.1429,0.0667,0.1333,0.9140,0.0643,15.0000,1.2649,1.4000,1.0198,3.0000,0.6325,0.2000,0.4000,"{'clf__max_depth': None, 'clf__min_samples_lea..."


shuffled-label shape: (6, 30)


,model,variant,roc_auc_mean,roc_auc_std,accuracy_mean,accuracy_std,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,balanced_accuracy_mean,balanced_accuracy_std,average_precision_mean,average_precision_std,f2_mean,f2_std,recall_positive_mean,recall_positive_std,recall_negative_mean,recall_negative_std,tn_mean,tn_std,fp_mean,fp_std,fn_mean,fn_std,tp_mean,tp_std
0,logistic_regression,shuffled_labels,0.4916,0.1900,0.7958,0.0462,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4754,0.0233,0.2470,0.0729,0.0000,0.0000,0.0000,0.0000,0.9507,0.0466,15.6000,1.0198,0.8000,0.7483,3.2000,0.4000,0.0000,0.0000
1,logistic_regression_balanced,shuffled_labels,0.4958,0.1997,0.5921,0.0703,0.2208,0.1604,0.1602,0.1084,0.3833,0.3317,0.5089,0.1688,0.2559,0.0743,0.2931,0.2320,0.3833,0.3317,0.6346,0.0647,10.4000,1.0198,6.0000,1.0954,2.0000,1.0954,1.2000,0.9798
2,random_forest,shuffled_labels,0.5126,0.1114,0.8068,0.0341,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4820,0.0147,0.2696,0.1137,0.0000,0.0000,0.0000,0.0000,0.9640,0.0294,15.8000,0.4000,0.6000,0.4899,3.2000,0.4000,0.0000,0.0000
3,random_forest_balanced,shuffled_labels,0.5404,0.1313,0.8268,0.0221,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4941,0.0118,0.3215,0.1731,0.0000,0.0000,0.0000,0.0000,0.9882,0.0235,16.2000,0.4000,0.2000,0.4000,3.2000,0.4000,0.0000,0.0000
4,svm_rbf,shuffled_labels,0.4282,0.1224,0.8368,0.0188,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000,0.0000,0.1875,0.0432,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,16.4000,0.4899,0.0000,0.0000,3.2000,0.4000,0.0000,0.0000
5,svm_rbf_balanced,shuffled_labels,0.4456,0.1485,0.6332,0.0851,0.1471,0.1232,0.1186,0.1026,0.2000,0.1633,0.4603,0.1014,0.2088,0.0767,0.1740,0.1429,0.2000,0.1633,0.7206,0.0954,11.8000,1.4697,4.6000,1.6248,2.6000,0.8000,0.6000,0.4899


stability shape: (36, 5)


,metric,scope,model,variant,value
0,spearman_rank_importance,cv_folds,logistic_regression,baseline_cv,0.6209
1,jaccard_top5_features,cv_folds,logistic_regression,baseline_cv,0.7333
2,mean_cv_shap_magnitude_across_features,cv_folds,logistic_regression,baseline_cv,0.5526
3,spearman_rank_importance,bootstrap,logistic_regression,baseline_bootstrap,0.0534
4,jaccard_top5_features,bootstrap,logistic_regression,baseline_bootstrap,0.3300
5,mean_cv_shap_magnitude_across_features,bootstrap,logistic_regression,baseline_bootstrap,0.5993
6,spearman_rank_importance,cv_folds,logistic_regression_balanced,baseline_cv,0.6855
7,jaccard_top5_features,cv_folds,logistic_regression_balanced,baseline_cv,0.6286
8,mean_cv_shap_magnitude_across_features,cv_folds,logistic_regression_balanced,baseline_cv,0.4885
9,spearman_rank_importance,bootstrap,logistic_regression_balanced,baseline_bootstrap,0.2818


## Analysis Helpers

In [4]:
def summarize_performance(df, metric_cols):
    missing = [col for col in metric_cols if col not in df.columns]
    if missing:
        raise ValueError(f'Missing expected columns: {missing}')
    return df[metric_cols].copy()

def compare_results(base_df, opt_df, compare_cols, index_col='model'):
    base_idx = base_df.set_index(index_col)[compare_cols].add_prefix('baseline_')
    opt_idx = opt_df.set_index(index_col)[compare_cols].add_prefix('optimized_')
    delta_df = base_idx.join(opt_idx, how='outer')
    for col in compare_cols:
        delta_df[f'delta_{col}'] = delta_df[f'optimized_{col}'] - delta_df[f'baseline_{col}']
    return delta_df

def compare_shuffled_control(real_df, shuffled_df, metric_cols, index_col='model'):
    real_idx = real_df.set_index(index_col)[metric_cols]
    shuffled_idx = shuffled_df.set_index(index_col)[metric_cols]
    joined = real_idx.join(shuffled_idx, lsuffix='_real', rsuffix='_shuffled', how='inner')
    joined['delta_roc_auc'] = joined['roc_auc_mean_real'] - joined['roc_auc_mean_shuffled']
    joined['delta_f1'] = joined['f1_mean_real'] - joined['f1_mean_shuffled']
    return joined

def pivot_stability(stability_df):
    return stability_df.pivot_table(
        index=['model', 'variant'],
        columns=['scope', 'metric'],
        values='value',
        aggfunc='first',
    )

def summarize_shap(df, label, top_n=15):
    if 'mean_abs_shap' in df.columns:
        sort_col = 'mean_abs_shap'
    else:
        numeric_cols = df.select_dtypes(include='number').columns.tolist()
        if not numeric_cols:
            raise ValueError(f'No numeric columns available to rank SHAP summary for {label}')
        sort_col = numeric_cols[-1]
    return df.sort_values(sort_col, ascending=False).head(top_n)

## Performance Summary

In [5]:
metric_cols = [
    'model',
    'variant',
    'roc_auc_mean',
    'roc_auc_std',
    'accuracy_mean',
    'f1_mean',
    'precision_mean',
    'recall_mean',
]

performance = pd.concat([
    summarize_performance(baseline, metric_cols),
    summarize_performance(optimized, metric_cols),
], ignore_index=True)

display(performance.sort_values(['roc_auc_mean', 'f1_mean'], ascending=False))

,model,variant,roc_auc_mean,roc_auc_std,accuracy_mean,f1_mean,precision_mean,recall_mean
9,svm_rbf_balanced,optimized,0.6468,0.1638,0.7121,0.0500,0.0500,0.0500
11,random_forest_balanced,optimized,0.6288,0.1380,0.7753,0.0800,0.1000,0.0667
10,random_forest,optimized,0.6157,0.1302,0.8063,0.0000,0.0000,0.0000
5,svm_rbf_balanced,baseline,0.5771,0.1586,0.7032,0.1111,0.1067,0.1167
2,random_forest,baseline,0.5196,0.1390,0.8063,0.0571,0.0667,0.0500
3,random_forest_balanced,baseline,0.5169,0.1726,0.8163,0.0000,0.0000,0.0000
4,svm_rbf,baseline,0.5162,0.1844,0.8368,0.0000,0.0000,0.0000
7,logistic_regression_balanced,optimized,0.5112,0.2793,0.6121,0.2067,0.1452,0.3667
6,logistic_regression,optimized,0.4958,0.2810,0.7968,0.0000,0.0000,0.0000
1,logistic_regression_balanced,baseline,0.4721,0.2817,0.6121,0.1844,0.1333,0.3167


## Baseline vs Optimized Delta

In [6]:
compare_cols = ['roc_auc_mean', 'accuracy_mean', 'f1_mean', 'precision_mean', 'recall_mean']
delta_df = compare_results(baseline, optimized, compare_cols)
display(delta_df.sort_values('delta_roc_auc_mean', ascending=False))

,baseline_roc_auc_mean,baseline_accuracy_mean,baseline_f1_mean,baseline_precision_mean,baseline_recall_mean,optimized_roc_auc_mean,optimized_accuracy_mean,optimized_f1_mean,optimized_precision_mean,optimized_recall_mean,delta_roc_auc_mean,delta_accuracy_mean,delta_f1_mean,delta_precision_mean,delta_recall_mean
model,,,,,,,,,,,,,,,
random_forest_balanced,0.5169,0.8163,0.0000,0.0000,0.0000,0.6288,0.7753,0.0800,0.1000,0.0667,0.1119,-0.0411,0.0800,0.1000,0.0667
random_forest,0.5196,0.8063,0.0571,0.0667,0.0500,0.6157,0.8063,0.0000,0.0000,0.0000,0.0961,0.0000,-0.0571,-0.0667,-0.0500
svm_rbf_balanced,0.5771,0.7032,0.1111,0.1067,0.1167,0.6468,0.7121,0.0500,0.0500,0.0500,0.0697,0.0089,-0.0611,-0.0567,-0.0667
logistic_regression,0.4298,0.7863,0.0000,0.0000,0.0000,0.4958,0.7968,0.0000,0.0000,0.0000,0.0660,0.0105,0.0000,0.0000,0.0000
logistic_regression_balanced,0.4721,0.6121,0.1844,0.1333,0.3167,0.5112,0.6121,0.2067,0.1452,0.3667,0.0392,0.0000,0.0222,0.0119,0.0500
svm_rbf,0.5162,0.8368,0.0000,0.0000,0.0000,0.4167,0.8368,0.0000,0.0000,0.0000,-0.0994,0.0000,0.0000,0.0000,0.0000


## Shuffled-Label Control

In [7]:
display(shuffled[metric_cols].sort_values('roc_auc_mean', ascending=False))

real_vs_shuffled = compare_shuffled_control(baseline[['model', 'roc_auc_mean', 'f1_mean']], shuffled[['model', 'roc_auc_mean', 'f1_mean']])
display(real_vs_shuffled.sort_values('delta_roc_auc', ascending=False))

,model,variant,roc_auc_mean,roc_auc_std,accuracy_mean,f1_mean,precision_mean,recall_mean
3,random_forest_balanced,shuffled_labels,0.5404,0.1313,0.8268,0.0000,0.0000,0.0000
2,random_forest,shuffled_labels,0.5126,0.1114,0.8068,0.0000,0.0000,0.0000
1,logistic_regression_balanced,shuffled_labels,0.4958,0.1997,0.5921,0.2208,0.1602,0.3833
0,logistic_regression,shuffled_labels,0.4916,0.1900,0.7958,0.0000,0.0000,0.0000
5,svm_rbf_balanced,shuffled_labels,0.4456,0.1485,0.6332,0.1471,0.1186,0.2000
4,svm_rbf,shuffled_labels,0.4282,0.1224,0.8368,0.0000,0.0000,0.0000


TypeError: compare_shuffled_control() missing 1 required positional argument: 'metric_cols'

## Stability Metrics

In [8]:
display(pivot_stability(stability))

scope                                                        baseline_vs_optimized                                                              bootstrap  \
metric                                          jaccard_top5_baseline_vs_optimized spearman_baseline_vs_optimized_global_importance jaccard_top5_features   
model                        variant                                                                                                                        
logistic_regression          baseline_bootstrap                                NaN                                              NaN                0.3300   
                             baseline_cv                                       NaN                                              NaN                   NaN   
                             comparison                                     0.6667                                           0.8529                   NaN   
logistic_regression_balanced baseline_bootstrap                                NaN                                              NaN                0.4077   
                             baseline_cv                                       NaN                                              NaN                   NaN   
                             comparison                                     1.0000                                           0.8529                   NaN   
random_forest                baseline_bootstrap                                NaN                                              NaN                0.4209   
                             baseline_cv                                       NaN                                              NaN                   NaN   
                             comparison                                     0.6667                                           0.9727                   NaN   
random_forest_balanced       baseline_bootstrap                                NaN                                              NaN                0.4265   
                             baseline_cv                                       NaN                                              NaN                   NaN   
                             comparison                                     0.6667                                           0.8273                   NaN   
svm_rbf                      comparison                                     0.4286                                           0.6364                   NaN   
svm_rbf_balanced             comparison                                     0.6667                                           0.9182                   NaN   

scope                                                                                                                        cv_folds                                         \
metric                                          mean_cv_shap_magnitude_across_features spearman_rank_importance jaccard_top5_features mean_cv_shap_magnitude_across_features   
model                        variant                                                                                                                                           
logistic_regression          baseline_bootstrap                                 0.5993                   0.0534                   NaN                                    NaN   
                             baseline_cv                                           NaN                      NaN                0.7333                                 0.5526   
                             comparison                                            NaN                      NaN                   NaN                                    NaN   
logistic_regression_balanced baseline_bootstrap                                 0.6495                   0.2818                   NaN                                    NaN   
                             baseline_cv                                           NaN            

## SHAP Output Checks

In [9]:
shap_baseline = load_csv_if_exists(files['shap_baseline'])
shap_optimized = load_csv_if_exists(files['shap_optimized'])

print('shap baseline shape:', shap_baseline.shape)
print('shap optimized shape:', shap_optimized.shape)

display(summarize_shap(shap_baseline, 'baseline'))
display(summarize_shap(shap_optimized, 'optimized'))

shap baseline shape: (6468, 5)
shap optimized shape: (6468, 5)


,participant_id,feature_name,shap_value,model,variant
1642,Process-rec-103,mean_clause_length,2.4708,logistic_regression_balanced,baseline
1422,Process-rec-066,mean_clause_length,2.1894,logistic_regression_balanced,baseline
1639,Process-rec-103,type_token_ratio,2.1825,logistic_regression_balanced,baseline
564,Process-rec-103,mean_clause_length,2.1164,logistic_regression,baseline
2040,Process-rec-147,noun_ratio,1.9993,logistic_regression_balanced,baseline
1862,Process-rec-127,mean_clause_length,1.9883,logistic_regression_balanced,baseline
344,Process-rec-066,mean_clause_length,1.8753,logistic_regression,baseline
1655,Process-rec-105,noun_ratio,1.8073,logistic_regression_balanced,baseline
561,Process-rec-103,type_token_ratio,1.7995,logistic_regression,baseline
1257,Process-rec-040,mean_clause_length,1.7069,logistic_regression_balanced,baseline


,participant_id,feature_name,shap_value,model,variant
1642,Process-rec-103,mean_clause_length,2.2110,logistic_regression_balanced,optimized
564,Process-rec-103,mean_clause_length,2.0980,logistic_regression,optimized
1422,Process-rec-066,mean_clause_length,1.9591,logistic_regression_balanced,optimized
344,Process-rec-066,mean_clause_length,1.8591,logistic_regression,optimized
1862,Process-rec-127,mean_clause_length,1.7792,logistic_regression_balanced,optimized
784,Process-rec-127,mean_clause_length,1.6883,logistic_regression,optimized
1639,Process-rec-103,type_token_ratio,1.6702,logistic_regression_balanced,optimized
1257,Process-rec-040,mean_clause_length,1.5274,logistic_regression_balanced,optimized
2040,Process-rec-147,noun_ratio,1.5159,logistic_regression_balanced,optimized
561,Process-rec-103,type_token_ratio,1.4698,logistic_regression,optimized


## Notes To Carry Forward

- These are development/Kaggle-style label results, not final ADReSS thesis results.
- Use AUROC and F1/recall together to detect overconfident low-recall models.
- Confirm that real-label performance exceeds shuffled-label control.
- Compare top SHAP features across baseline and optimized models for consistency.